# Manifest explorer

The generator does not write audio. It writes a **manifest**: one CSV row per trial, holding every random choice that trial needs - which speakers, which recordings, when each one starts talking, how loud, which room, which noise clip. A renderer reads a row and produces exactly one mixture from it.

This notebook reads a manifest and explains what is in it, one theme at a time, so the recipe is understood before any audio exists.

Point `SPLIT` at any manifest in `data/manifests/` and re-run. Nothing below is specific to one split.

## 0. Setup

Loads four things for the chosen split:

| | |
|---|---|
| **manifest** | `data/manifests/<split>.csv` — the trials |
| **meta** | `<split>.meta.yaml` — when it was built, from which commit and seed |
| **generator settings** | the `defaults` + this split's overrides from the config, so we can check the manifest against the ranges it was told to sample |
| **indexes** | `data/index/` — every LibriSpeech utterance (speaker, duration, transcript) and every noise clip. The manifest stores only IDs, so durations and text are looked up here |

In [2]:
# Which manifest to analyse. Works for smoke_train, smoke_val, train, val,
# eval_public, eval_private, or a scratch manifest via MANIFEST_PATH.

SPLIT = "smoke_train"
MANIFEST_PATH = None      # full path to a .csv; overrides SPLIT when set

In [3]:
from pathlib import Path
from types import SimpleNamespace
import hashlib

import numpy as np
import pandas as pd
import yaml

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 100)

# The generator packs multi-value fields as "a|b|c". These become real lists.
LIST_COLS = {
    "target_utts": "target_utt_list",
    "target_onsets_s": "target_onset_list",
    "interferer_utts": "interferer_utt_list",
    "interferer_onsets_s": "interferer_onset_list",
}


def find_root(start=None):
    """Walk up until CLAUDE.md appears, so paths resolve whether the kernel
    started in src/exploratory/ or at the repo root."""
    here = (start or Path.cwd()).resolve()
    for d in [here, *here.parents]:
        if (d / "CLAUDE.md").exists():
            return d
    raise RuntimeError(f"repo root not found above {here}")


ROOT = find_root()


def _as_list(cell):
    """"a|b" -> ["a", "b"]. Empty or missing -> []. Target-absent trials
    genuinely have no target utterances, so [] is a real value, not a bug."""
    if not isinstance(cell, str) or not cell.strip():
        return []
    return cell.split("|")


def load(split, manifest_path=None):
    """Everything needed to analyse one split, bundled together."""
    man = Path(manifest_path) if manifest_path else ROOT / "data/manifests" / f"{split}.csv"
    # Speaker and chapter IDs are numeric strings; without dtype=str pandas
    # turns them into ints and "0090" stops matching the folder on disk.
    df = pd.read_csv(man, dtype={
        "target_speaker": str, "interferer_speaker": str,
        "target_chapter": str, "interferer_chapter": str,
    })
    columns = list(df.columns)

    for raw, parsed in LIST_COLS.items():
        vals = df[raw].map(_as_list)
        df[parsed] = vals if raw.endswith("_utts") else vals.map(
            lambda v: [float(x) for x in v])

    meta = yaml.safe_load(man.with_suffix(".meta.yaml").read_text())

    cfg_file = ROOT / meta["config"]
    config = yaml.safe_load(cfg_file.read_text())
    cfg = {**config["defaults"], **config["splits"].get(split, {})}
    cfg["seed"], cfg["sample_rate"] = config["seed"], config["sample_rate"]
    # If the config changed after the manifest was written, the ranges we check
    # against are not the ranges that were actually sampled from.
    stale = hashlib.md5(cfg_file.read_bytes()).hexdigest() != meta.get("config_md5")

    idx = ROOT / "data/index"
    utt_file = idx / f"utterances_{split}.csv"
    utts = (pd.read_csv(utt_file, dtype={"speaker": str, "chapter": str}).set_index("utt")
            if utt_file.exists() else None)
    noise_file = idx / f"noise_{cfg['noise_split']}.csv"
    noise = pd.read_csv(noise_file).set_index("clip") if noise_file.exists() else None

    # Onsets alone do not say when a speaker stops. Attach each chosen
    # utterance's length so talking intervals can be reconstructed.
    if utts is not None:
        dur = utts["duration"].to_dict()
        for who in ("target", "interferer"):
            df[f"{who}_dur_list"] = df[f"{who}_utt_list"].map(
                lambda us: [dur[u] for u in us])

    return SimpleNamespace(split=split, path=man, df=df, columns=columns,
                           meta=meta, cfg=cfg, utts=utts, noise=noise,
                           config_stale=stale)


def spans(row, who):
    """When `who` is actually talking, as [(start_s, end_s), ...]."""
    return [(o, o + d) for o, d in
            zip(row[f"{who}_onset_list"], row[f"{who}_dur_list"])]


def text_of(D, row, who):
    """The words spoken by `who` in this trial, in order."""
    return " ".join(D.utts.loc[u, "text"] for u in row[f"{who}_utt_list"])

In [4]:
D = load(SPLIT, MANIFEST_PATH)

print(f"{D.path.relative_to(ROOT)}")
print(f"  {len(D.df)} trials, {len(D.columns)} columns")
print(f"  built {D.meta['generated']}  seed {D.meta['seed']}  "
      f"commit {D.meta['git_commit'][:7]}  unsatisfiable {D.meta['n_failed']}")
print(f"  generator {D.meta['config']}"
      + ("   ** CHANGED since build — ranges below may not match **"
         if D.config_stale else "   (unchanged since build)"))
print(f"  utterance index  {'loaded' if D.utts is not None else 'MISSING'}"
      f"   ({len(D.utts)} utterances)" if D.utts is not None else "")
print(f"  noise index      {'loaded' if D.noise is not None else 'MISSING'}"
      f"   ({len(D.noise)} clips, WHAM! split '{D.cfg['noise_split']}')"
      if D.noise is not None else "")

data/manifests/smoke_train.csv
  50 trials, 41 columns
  built 2026-08-10  seed 42  commit 9d1c68b  unsatisfiable 0
  generator experiments/configs/generator.yaml   (unchanged since build)
  utterance index  loaded   (2239 utterances)
  noise index      loaded   (20000 clips, WHAM! split 'tr')


## 1. What each row does

### 1.1  `trial_id`

Each trial generated by the manifest can be replaced/regenerated. Secondly, since the manifest made use of `blake2b` instead of `hash()`, we are guaranteed to get identical data between different authors/users of the repository.

In [8]:
D.df.head(10)

,trial_id,split,target_speaker,target_chapter,target_utts,target_onsets_s,target_speech_s,target_activity,interferer_speaker,interferer_chapter,interferer_utts,interferer_onsets_s,interferer_speech_s,interferer_activity,enrollment_utt,enrollment_offset_s,enrollment_length_s,enrollment_eq,noise_clip,noise_offset_s,mixture_length_s,sir_db,snr_db,target_loudness_lufs,overlap_requested,overlap_achieved,t60_s,room_l,room_w,room_h,mic_x,mic_y,mic_z,target_x,target_y,target_z,interferer_x,interferer_y,interferer_z,target_absent,same_gender,target_utt_list,target_onset_list,interferer_utt_list,interferer_onset_list,target_dur_list,interferer_dur_list
0,smoke_train-42-000000,smoke_train,242,NaN,NaN,NaN,0.000,0.0000,90,130566,90-130566-0002,0.4083,14.805,0.8414,242-126842-0001,1.7670,5.0,1,014o0306_1.9838_01io030c_-1.9838.flac,1.4199,17.595,NaN,19.01,-28.02,0.0000,0.0000,0.2406,8.130,5.650,3.687,4.934,3.131,1.416,5.456,3.543,1.496,4.609,5.001,1.671,1,0,[],[],[90-130566-0002],[],[],[14.805]
1,smoke_train-42-000001,smoke_train,441,NaN,NaN,NaN,0.000,0.0000,98,199,98-199-0020,3.1295,15.725,0.8234,441-130108-0034,0.3469,5.0,0,40fa0102_1.6539_20ec0104_-1.6539.flac,6.9748,19.097,NaN,6.95,-31.63,0.0000,0.0000,0.4097,8.516,6.578,3.578,5.015,3.034,1.419,3.903,2.925,1.137,3.359,3.222,1.192,1,1,[],[],[98-199-0020],[],[],[15.725]
2,smoke_train-42-000002,smoke_train,8580,287364,8580-287364-0015,3.9755,13.650,0.7595,441,130108,441-130108-0027,3.3058,5.290,0.2943,8580-287363-0002,2.9080,5.0,1,20bc0104_0.75061_01gc0207_-0.75061.flac,7.0969,17.973,2.70,5.88,-25.16,0.2571,0.2571,0.2449,8.049,6.792,3.489,2.927,3.381,1.727,2.466,2.564,1.373,3.559,2.063,1.740,0,0,[8580-287364-0015],[3.9755],[441-130108-0027],[],[13.65],[5.29]
3,smoke_train-42-000003,smoke_train,6497,234100,6497-234100-0035,0.3103,13.505,0.7559,98,121658,98-121658-0015,3.3946,10.545,0.5902,6497-234106-0011,0.6604,5.0,0,204a010y_0.87783_018a0108_-0.87783.flac,7.9055,17.866,11.23,19.39,-31.02,0.5833,0.5833,0.2500,8.472,5.672,3.560,3.689,2.302,1.099,4.429,1.415,1.767,2.393,2.289,1.356,0,0,[6497-234100-0035],[0.3103],[98-121658-0015],[],[13.505],[10.545]
4,smoke_train-42-000004,smoke_train,227,NaN,NaN,NaN,0.000,0.0000,90,130566,90-130566-0016,2.8518,14.280,0.8097,227-129974-0046,7.0523,5.0,0,01wo030l_0.38677_01lo0319_-0.38677.flac,15.2530,17.635,NaN,9.17,-31.29,0.0000,0.0000,0.4007,8.577,8.728,3.382,3.587,3.158,0.943,4.504,3.092,1.257,2.475,3.853,1.215,1,0,[],[],[90-130566-0016],[],[],[14.28]
5,smoke_train-42-000005,smoke_train,8580,287364,8580-287364-0022,2.4897,14.830,0.7810,1054,143005,1054-143005-0103,0.0000,8.460,0.4456,8580-287363-0003,2.6152,5.0,0,404o030f_0.095296_01pa010o_-0.095296.flac,1.7790,18.987,1.96,15.37,-29.56,0.2915,0.3144,0.5953,5.963,8.377,3.258,3.833,3.961,1.577,5.024,4.307,1.439,1.991,3.954,1.663,0,0,[8580-287364-0022],[2.4897],[1054-143005-0103],[],[14.83],[8.46]
6,smoke_train-42-000006,smoke_train,6553,NaN,NaN,NaN,0.000,0.0000,227,129974,227-129974-0036,0.4796,13.670,0.8488,6553-86683-0075,0.1842,5.0,0,011c021b_2.1436_018a0111_-2.1436.flac,3.0664,16.104,NaN,13.04,-32.46,0.0000,0.0000,0.1508,8.950,8.372,3.121,4.125,4.302,1.474,5.423,2.809,1.476,5.421,4.758,1.118,1,0,[],[],[227-129974-0036],[],[],[13.67]
7,smoke_train-42-000007,smoke_train,6446,40544,6446-40544-0013,1.5155,11.855,0.7720,5246,30101,5246-30101-0001,8.2166,7.140,0.4649,6446-40571-0026,1.5446,5.0,1,20to0104_1.8698_20ca010p_-1.8698.flac,2.4702,15.357,3.50,2.02,-26.06,0.3136,0.3356,0.3840,6.333,7.027,3.902,2.861,3.402,1.764,4.161,4.743,1.392,1.312,4.574,1.132,0,1,[6446-40544-0013],[1.5155],[5246-30101-0001],[],[11.855],[7.14]
8,smoke_train-42-000008,smoke_train,6497,234100,6497-234100-0027,0.8353,16.330,0.8194,441,130108,441-130108-0034,9.0698,10.355,0.5196,6497-234067-0031,2.1752,5.0,1,01qc020f_0.39027_20oo0105_-0.39027.flac,11.5947,19.929,4.05,10.83,-32.13,0.4059,0.4062,0.2586,9.868,7.986,3.554,3.945,3.755,1.310,2.911,4.248,1.489,2.563,3.441,1.372,0,0,[6497-234100-0027],[0.8353],[441-130108-0034],[],[16.33],[10.

In [9]:
print(f"  target speaker IDs: {D.df['target_speaker'].nunique()} "
      f"({D.df['target_speaker'].unique()})")

  target speaker IDs: 17 (<StringArray>
['242', '441', '8580', '6497', '227', '6553', '6446', '834', '3630', '90', '7832', '7739', '98', '233', '5246', '7264', '6476']
Length: 17, dtype: str)


In [12]:
D.df.columns

Index(['trial_id', 'split', 'target_speaker', 'target_chapter', 'target_utts', 'target_onsets_s', 'target_speech_s', 'target_activity', 'interferer_speaker', 'interferer_chapter', 'interferer_utts',
       'interferer_onsets_s', 'interferer_speech_s', 'interferer_activity', 'enrollment_utt', 'enrollment_offset_s', 'enrollment_length_s', 'enrollment_eq', 'noise_clip', 'noise_offset_s',
       'mixture_length_s', 'sir_db', 'snr_db', 'target_loudness_lufs', 'overlap_requested', 'overlap_achieved', 't60_s', 'room_l', 'room_w', 'room_h', 'mic_x', 'mic_y', 'mic_z', 'target_x',
       'target_y', 'target_z', 'interferer_x', 'interferer_y', 'interferer_z', 'target_absent', 'same_gender', 'target_utt_list', 'target_onset_list', 'interferer_utt_list', 'interferer_onset_list',
       'target_dur_list', 'interferer_dur_list'],
      dtype='str')